# Stage 2: Fingerprinty i przygotowanie danych treningowych

Ten notatnik:
1. Wczytuje dane po etapie 1.
2. Buduje fingerprinty (`ECFP`, `MACCS`).
3. Tworzy split oparty o scaffolds (Murcko).
4. Tworzy macierze hierarchii klas z pliku OBO.
5. Zapisuje artefakty do dalszego treningu modelu.

In [1]:
from pathlib import Path
import json
import re
from typing import Dict, List, Set

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_selection import chi2
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint, AtomPairFingerprint

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data/chebi_dataset_train_stage1.parquet"
OBO_PATH = PROJECT_ROOT / "extras/chebi_classes.obo"
DEFS_PATH = PROJECT_ROOT / "extras/chebi_class_definitions.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "data/stage2_artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_RE = re.compile(r"^class_(\d+)$")

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Nieobslugiwany format: {suffix}")


def find_class_columns(columns: List[str]) -> List[str]:
    pairs = []
    for c in columns:
        m = CLASS_RE.match(c)
        if m:
            pairs.append((int(m.group(1)), c))
    pairs.sort(key=lambda x: x[0])
    return [c for _, c in pairs]


def parse_obo_parents(obo_path: Path) -> Dict[str, Set[str]]:
    parents: Dict[str, Set[str]] = {}
    current_id = None
    with obo_path.open("r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.strip()
            if line == "[Term]":
                current_id = None
                continue
            if line.startswith("id: "):
                current_id = line[4:].strip()
                parents.setdefault(current_id, set())
                continue
            if line.startswith("is_a: ") and current_id:
                parent_id = line[6:].split("!")[0].strip()
                if parent_id:
                    parents[current_id].add(parent_id)
    return parents


def compute_ancestors(class_cols: List[str], parent_map: Dict[str, Set[str]]) -> Dict[str, Set[str]]:
    allowed = set(class_cols)
    memo: Dict[str, Set[str]] = {}

    def dfs(node: str, stack: Set[str] | None = None) -> Set[str]:
        if node in memo:
            return memo[node]
        if stack is None:
            stack = set()
        if node in stack:
            return set()
        stack = set(stack)
        stack.add(node)

        direct = {p for p in parent_map.get(node, set()) if p in allowed}
        out = set(direct)
        for p in direct:
            out.update(dfs(p, stack))
        memo[node] = out
        return out

    for cls in class_cols:
        dfs(cls)
    return memo


def scaffold_from_smiles(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)


def standardize_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Szybka kanonikalizacja bez kosztownej normalizacji chemicznej.
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

In [3]:
df = read_table(DATA_PATH)
class_cols = find_class_columns(df.columns.tolist())

required = {"mol_id", "SMILES"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Brakuje kolumn wymaganych: {missing_required}")

if not class_cols:
    raise ValueError("Nie znaleziono kolumn class_i")

if "is_valid_smiles" in df.columns:
    df = df[df["is_valid_smiles"].astype(bool)].copy()

# Step 2: standardized canonical chemistry representation
df = df[df["SMILES"].notna()].copy()
df["SMILES"] = df["SMILES"].astype(str)
df["standardized_smiles"] = df["SMILES"].map(standardize_smiles)
df = df[df["standardized_smiles"].notna()].copy()

# Deduplicate by standardized structure, keep first row
df = df.drop_duplicates(subset=["standardized_smiles"], keep="first").reset_index(drop=True)

smiles_col = "standardized_smiles"
for c in class_cols:
    df[c] = df[c].astype(bool)

print("Liczba rekordow:", len(df))
print("Liczba klas:", len(class_cols))
print("Kolumna SMILES do cech:", smiles_col)
df[["mol_id", "SMILES", "standardized_smiles"] + class_cols[:5]].head(3)

[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] WARNING: not removing hydrogen atom without neighbors
[22:20:13] Unusual charge on atom 0 number of radical electrons set to zero
[22:20:14] WARNING: not removing hydrogen atom without neighbors
[22:20:14] WARNING: not removing hydrogen atom without neighbors
[22:20:14] WARNING: not removing hydrogen atom without neighbors
[22:20:14] WARNING: not removing hydrogen atom without neighbors
[22:20:14] WARNING: not removing hydrogen atom without neighbors
[22:20:14] WAR

Liczba rekordow: 33631
Liczba klas: 500
Kolumna SMILES do cech: standardized_smiles


,mol_id,SMILES,standardized_smiles,class_0,class_1,class_2,class_3,class_4
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,CCCCC/C=C\CCCCCCCC(=O)O,True,True,True,True,True
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,Cc1cc2cc(O)cc(O)c2c(C)n1,True,True,True,True,True
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,True,True,True,True,True


In [4]:
df.head()

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_494,class_495,class_496,class_497,class_498,class_499,is_valid_smiles,canonical_smiles,inchikey,standardized_smiles
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,True,CCCCC/C=C\CCCCCCCC(=O)O,DJCQJZKZUCHHAL-SREVYHEPSA-N,CCCCC/C=C\CCCCCCCC(=O)O
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,True,Cc1cc2cc(O)cc(O)c2c(C)n1,NFCOBHKSUZDTLE-UHFFFAOYSA-N,Cc1cc2cc(O)cc(O)c2c(C)n1
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,True,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,JUGXQEJPWDYOJV-YSQMORBQSA-N,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,True,True,True,True,True,True,True,False,...,False,False,False,False,False,False,True,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,CNMRNUFFAOLBLH-UHFFFAOYSA-N,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,True,True,True,True,True,True,True,True,...,False,False,False,False,False,False,True,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,UXWZMQPNSYWAHX-DMELVVMMSA-N,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...


In [5]:
# Fingerprinty + cechy ciagle (po standaryzacji)
smiles = df[smiles_col].tolist()
mols = [Chem.MolFromSmiles(s) for s in smiles]

ecfp = ECFPFingerprint(fp_size=2048, radius=2, include_chirality=True, n_jobs=-1)
maccs = MACCSFingerprint(n_jobs=-1)
atom_pair = AtomPairFingerprint(fp_size=2048, n_jobs=-1)

X_ecfp_raw = ecfp.transform(smiles).astype(np.uint8)
X_maccs_raw = maccs.transform(smiles).astype(np.uint8)
X_atom_pair_raw = atom_pair.transform(smiles).astype(np.uint8)

desc_names = [
    "MolWt", "TPSA", "MolLogP", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "RingCount", "FractionCSP3", "HeavyAtomCount",
    "NHOHCount", "NOCount", "NumValenceElectrons",
    "NumAtoms", "NumBonds", "AromaticAtomRatio",
]


def compute_desc(mol):
    if mol is None:
        return [np.nan] * len(desc_names)
    n_atoms = mol.GetNumAtoms()
    n_bonds = mol.GetNumBonds()
    aromatic_atoms = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
    aromatic_ratio = aromatic_atoms / max(n_atoms, 1)
    return [
        Descriptors.MolWt(mol),
        Descriptors.TPSA(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol),
        Descriptors.NHOHCount(mol),
        Descriptors.NOCount(mol),
        Descriptors.NumValenceElectrons(mol),
        n_atoms,
        n_bonds,
        aromatic_ratio,
    ]


X_cont_raw = np.array([compute_desc(m) for m in mols], dtype=np.float32)
Y = df[class_cols].astype(np.uint8).to_numpy()

print("X_ecfp_raw:", X_ecfp_raw.shape, X_ecfp_raw.dtype)
print("X_maccs_raw:", X_maccs_raw.shape, X_maccs_raw.dtype)
print("X_atom_pair_raw:", X_atom_pair_raw.shape, X_atom_pair_raw.dtype)
print("X_cont_raw:", X_cont_raw.shape, X_cont_raw.dtype)
print("Y:", Y.shape, Y.dtype)

[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] Unusual charge on atom 0 number of radical electrons set to zero
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WARNING: not removing hydrogen atom without neighbors
[22:20:20] WAR

X_ecfp_raw: (33631, 2048) uint8
X_maccs_raw: (33631, 166) uint8
X_atom_pair_raw: (33631, 2048) uint8
X_cont_raw: (33631, 15) float32
Y: (33631, 500) uint8


In [6]:
# Hierarchia klas (macierze parent/ancestor)
parent_map = parse_obo_parents(OBO_PATH)
anc_map = compute_ancestors(class_cols, parent_map)

class_idx = {c: i for i, c in enumerate(class_cols)}
C = len(class_cols)
M_parent = np.zeros((C, C), dtype=np.uint8)
M_ancestor = np.zeros((C, C), dtype=np.uint8)

for child, parents in parent_map.items():
    if child not in class_idx:
        continue
    i = class_idx[child]
    for p in parents:
        if p in class_idx:
            M_parent[i, class_idx[p]] = 1

for child, ancestors in anc_map.items():
    if child not in class_idx:
        continue
    i = class_idx[child]
    for a in ancestors:
        if a in class_idx:
            M_ancestor[i, class_idx[a]] = 1

print("M_parent:", M_parent.shape, "edges:", int(M_parent.sum()))
print("M_ancestor:", M_ancestor.shape, "edges:", int(M_ancestor.sum()))

M_parent: (500, 500) edges: 748
M_ancestor: (500, 500) edges: 8610


In [7]:
# Split train/valid po scaffoldach (mniejszy leakage)
scaffolds = np.array([scaffold_from_smiles(s) for s in smiles])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx = np.arange(len(df))
train_idx, valid_idx = next(gss.split(idx, groups=scaffolds))


def prevalence_filter_mask(X_bin: np.ndarray, train_idx_arr: np.ndarray, low: float = 0.005, high: float = 0.40):
    p = X_bin[train_idx_arr].mean(axis=0)
    return (p >= low) & (p <= high)


def chi2_topk_mask(X_bin: np.ndarray, y_bin: np.ndarray, train_idx_arr: np.ndarray, topk: int):
    topk = int(min(topk, X_bin.shape[1]))
    if topk <= 0:
        return np.zeros(X_bin.shape[1], dtype=bool)
    Xtr = X_bin[train_idx_arr]
    Ytr = y_bin[train_idx_arr]
    scores_max = np.zeros(Xtr.shape[1], dtype=np.float64)
    for c in range(Ytr.shape[1]):
        y = Ytr[:, c]
        if np.unique(y).size < 2:
            continue
        scores, _ = chi2(Xtr, y)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
        scores_max = np.maximum(scores_max, scores)
    keep_local = np.zeros(Xtr.shape[1], dtype=bool)
    top_idx = np.argpartition(scores_max, -topk)[-topk:]
    keep_local[top_idx] = True
    return keep_local


# 1) Prevalence filter -> 2) Chi2 top-k (na train)
ecfp_prev = prevalence_filter_mask(X_ecfp_raw, train_idx, low=0.005, high=0.40)
maccs_prev = prevalence_filter_mask(X_maccs_raw, train_idx, low=0.001, high=0.80)
atom_pair_prev = prevalence_filter_mask(X_atom_pair_raw, train_idx, low=0.002, high=0.40)

ecfp_chi = chi2_topk_mask(X_ecfp_raw[:, ecfp_prev], Y, train_idx, topk=1024)
maccs_chi = chi2_topk_mask(X_maccs_raw[:, maccs_prev], Y, train_idx, topk=min(96, int(maccs_prev.sum())))
atom_pair_chi = chi2_topk_mask(X_atom_pair_raw[:, atom_pair_prev], Y, train_idx, topk=1024)

ecfp_mask = np.zeros(X_ecfp_raw.shape[1], dtype=bool)
maccs_mask = np.zeros(X_maccs_raw.shape[1], dtype=bool)
atom_pair_mask = np.zeros(X_atom_pair_raw.shape[1], dtype=bool)

ecfp_mask[np.where(ecfp_prev)[0][ecfp_chi]] = True
maccs_mask[np.where(maccs_prev)[0][maccs_chi]] = True
atom_pair_mask[np.where(atom_pair_prev)[0][atom_pair_chi]] = True

X_ecfp = X_ecfp_raw[:, ecfp_mask].astype(np.uint8)
X_maccs = X_maccs_raw[:, maccs_mask].astype(np.uint8)
X_atom_pair = X_atom_pair_raw[:, atom_pair_mask].astype(np.uint8)

# Cechy ciagle: clipping (train quantiles) + correlation filter + robust scaling
X_cont_filled = np.nan_to_num(X_cont_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
q_low = np.percentile(X_cont_filled[train_idx], 1.0, axis=0)
q_high = np.percentile(X_cont_filled[train_idx], 99.0, axis=0)
X_cont_clip = np.clip(X_cont_filled, q_low, q_high)

# remove near-constant
std_train = X_cont_clip[train_idx].std(axis=0)
cont_nonconst_mask = std_train > 1e-8
X_cont_clip = X_cont_clip[:, cont_nonconst_mask]
desc_names_kept = [d for d, m in zip(desc_names, cont_nonconst_mask) if m]

# correlation filter on train
corr = np.corrcoef(X_cont_clip[train_idx], rowvar=False)
corr = np.nan_to_num(corr, nan=0.0)
keep = np.ones(corr.shape[0], dtype=bool)
for i in range(corr.shape[0]):
    if not keep[i]:
        continue
    for j in range(i + 1, corr.shape[0]):
        if keep[j] and abs(corr[i, j]) > 0.98:
            keep[j] = False

X_cont_sel = X_cont_clip[:, keep]
desc_names_sel = [d for d, m in zip(desc_names_kept, keep) if m]

cont_scaler = RobustScaler()
cont_scaler.fit(X_cont_sel[train_idx])
X_cont = cont_scaler.transform(X_cont_sel).astype(np.float32)

print("Train:", len(train_idx), "Valid:", len(valid_idx))
print("Unikalne scaffolds (train):", len(set(scaffolds[train_idx])))
print("Unikalne scaffolds (valid):", len(set(scaffolds[valid_idx])))
print("ECFP bits po filtracji:", X_ecfp.shape[1], "z", X_ecfp_raw.shape[1])
print("MACCS bits po filtracji:", X_maccs.shape[1], "z", X_maccs_raw.shape[1])
print("AtomPair bits po filtracji:", X_atom_pair.shape[1], "z", X_atom_pair_raw.shape[1])
print("X_cont po selekcji+skalowaniu:", X_cont.shape)
print("Cechy ciagle zachowane:", desc_names_sel)

[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] WARNING: not removing hydrogen atom without neighbors
[22:20:52] Unusual charge on atom 0 number of radical electrons set to zero
[22:20:53] WARNING: not removing hydrogen atom without neighbors
[22:20:53] WARNING: not removing hydrogen atom without neighbors
[22:20:53] WARNING: not removing hydrogen atom without neighbors
[22:20:53] WARNING: not removing hydrogen atom without neighbors
[22:20:53] WARNING: not removing hydrogen atom without neighbors
[22:20:53] WAR

Train: 20682 Valid: 12949
Unikalne scaffolds (train): 5850
Unikalne scaffolds (valid): 1463
ECFP bits po filtracji: 1024 z 2048
MACCS bits po filtracji: 96 z 166
AtomPair bits po filtracji: 1024 z 2048
X_cont po selekcji+skalowaniu: (33631, 9)
Cechy ciagle zachowane: ['MolWt', 'TPSA', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'RingCount', 'FractionCSP3', 'AromaticAtomRatio']


In [10]:
# Zapis artefaktow
artifact_npz = ARTIFACTS_DIR / "stage2_fingerprints.npz"
artifact_meta = ARTIFACTS_DIR / "stage2_fingerprints_meta.json"
artifact_index = ARTIFACTS_DIR / "stage2_row_index.parquet"

np.savez_compressed(
    artifact_npz,
    X_ecfp=X_ecfp,
    X_maccs=X_maccs,
    X_atom_pair=X_atom_pair,
    X_cont=X_cont,
    Y=Y,
    train_idx=train_idx,
    valid_idx=valid_idx,
    M_parent=M_parent,
    M_ancestor=M_ancestor,
    ecfp_mask=ecfp_mask.astype(np.uint8),
    maccs_mask=maccs_mask.astype(np.uint8),
    atom_pair_mask=atom_pair_mask.astype(np.uint8),
    cont_low=q_low.astype(np.float32),
    cont_high=q_high.astype(np.float32),
    cont_nonconst_mask=cont_nonconst_mask.astype(np.uint8),
    cont_corr_keep_mask=keep.astype(np.uint8),
    cont_center=cont_scaler.center_.astype(np.float32),
    cont_scale=cont_scaler.scale_.astype(np.float32),
)

index_cols = ["mol_id", "SMILES", smiles_col]
if "inchikey" in df.columns:
    index_cols.append("inchikey")
df[index_cols].reset_index(drop=True).to_parquet(artifact_index, index=False)

class_names = {}
if DEFS_PATH.exists():
    defs = pd.read_csv(DEFS_PATH)
    if {"chebi_id", "name"}.issubset(defs.columns):
        class_names = dict(zip(defs["chebi_id"], defs["name"]))

meta = {
    "data_path": str(DATA_PATH),
    "n_rows": int(len(df)),
    "n_classes": int(len(class_cols)),
    "smiles_column": smiles_col,
    "ecfp_shape": list(X_ecfp.shape),
    "maccs_shape": list(X_maccs.shape),
    "atom_pair_shape": list(X_atom_pair.shape),
    "continuous_shape": list(X_cont.shape),
    "continuous_feature_names": desc_names_sel,
    "target_shape": list(Y.shape),
    "train_size": int(len(train_idx)),
    "valid_size": int(len(valid_idx)),
    "noise_filter": {
        "ecfp": {"prevalence_low": 0.005, "prevalence_high": 0.40, "topk": 1024, "kept": int(ecfp_mask.sum())},
        "maccs": {"prevalence_low": 0.001, "prevalence_high": 0.80, "topk": min(96, int(maccs_prev.sum())), "kept": int(maccs_mask.sum())},
        "atom_pair": {"prevalence_low": 0.002, "prevalence_high": 0.40, "topk": 1024, "kept": int(atom_pair_mask.sum())},
    },
    "continuous_processing": {
        "clip_percentiles": [1.0, 99.0],
        "corr_threshold": 0.98,
        "scaler": "RobustScaler",
    },
    "class_columns": class_cols,
    "class_names": {c: class_names.get(c, "") for c in class_cols},
    "files": {
        "npz": str(artifact_npz),
        "meta_json": str(artifact_meta),
        "row_index": str(artifact_index),
    },
}

with artifact_meta.open("w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Zapisano:")
print("-", artifact_npz)
print("-", artifact_meta)
print("-", artifact_index)

Zapisano:
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints.npz
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints_meta.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_row_index.parquet


In [11]:
# Szybki podgląd częstości klas (top-10)
freq = pd.Series(Y.mean(axis=0), index=class_cols).sort_values(ascending=False)
top10 = freq.head(10).to_frame("positive_fraction")
top10

,positive_fraction
class_0,1.000000
class_1,0.994172
class_2,0.974577
class_3,0.962951
class_4,0.923136
class_5,0.921233
class_6,0.823585
class_7,0.743956
class_8,0.720972
class_9,0.695846
